# Setup and Auth

Use this notebook to verify imports, configure the nmrXiv API base URL, and try the authentication endpoints from the dev Swagger docs.

## Authentication endpoints covered

- `POST /api/auth/login` - authenticate user and generate a Bearer token.
- `GET /api/auth/logout` - revoke the current Bearer token.
- `POST /api/auth/register` - register a new user account.
- `GET /api/auth/user/info` - fetch the current authenticated user.
- `GET /api/email/verify/{user_id}/{hash}` - verify a user email address.
- `GET /api/auth/email/resend` - resend the email verification link.

In [2]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://dev.nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
})

BASE_URL, API_BASE

('https://dev.nmrxiv.org', 'https://dev.nmrxiv.org/api')

In [4]:
def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def extract_token(payload):
    if not isinstance(payload, dict):
        return None
    token = payload.get("access_token") or payload.get("token")
    if token:
        return token
    data = payload.get("data")
    if isinstance(data, dict):
        return data.get("access_token") or data.get("token")
    return None


def require_bearer_token():
    if "Authorization" not in session.headers:
        raise RuntimeError("Login first so the session has an Authorization: Bearer <token> header.")

## Authenticate user and generate access token

Authenticates a user with email and password credentials, returns a Bearer token for API access. Email verification is required for successful authentication. Requires `NMRXIV_EMAIL` and `NMRXIV_PASSWORD` in `.env`. The login response returns an `access_token`; the notebook stores it only in this in-memory `session`.

In [5]:
email = os.getenv("NMRXIV_EMAIL")
password = os.getenv("NMRXIV_PASSWORD")

if email and password:
    login_response = api_request("POST", "/auth/login", json={"email": email, "password": password})
    token = extract_token(login_response)
    if token:
        session.headers.update({"Authorization": f"Bearer {token}"})
        print("Bearer token configured for this notebook session.")
    else:
        print("Login succeeded, but no token key was recognized. Inspect login_response if needed.")
else:
    print("Set NMRXIV_EMAIL and NMRXIV_PASSWORD in .env to run login.")

POST https://dev.nmrxiv.org/api/auth/login -> 200
Bearer token configured for this notebook session.


## Get current authenticated user information

Retrieves detailed information about the currently authenticated user including profile data, team memberships, roles, and account status. Requires valid Bearer token authentication.

In [6]:
if "Authorization" in session.headers:
    user_info = api_request("GET", "/auth/user/info")
    pprint(user_info)
else:
    print("Login first to call /auth/user/info.")

GET https://dev.nmrxiv.org/api/auth/user/info -> 200
{'affiliation': None,
 'created_at': '2026-01-22T10:21:45.000000Z',
 'current_team_id': 1,
 'email': 'nisha.sharma@uni-jena.de',
 'email_verified_at': '2026-01-27T10:26:42.000000Z',
 'first_name': 'Nisha',
 'id': 1,
 'last_name': 'Sharma',
 'name': 'Nisha Sharma',
 'onboarded': True,
 'orcid_id': None,
 'primed': False,
 'profile_photo_path': None,
 'profile_photo_url': 'https://ui-avatars.com/api/?name=Nisha%2BSharma&color=7F9CF5&background=EBF4FF',
 'ror_id': None,
 'updated_at': '2026-01-27T10:26:47.000000Z',
 'username': 'nis',
 'welcome_valid_until': None}


## Verify user email address

Verifies a user's email address using a signed URL sent via email. This endpoint processes the verification link and marks the user's email as verified. Users must click the verification link sent to their email inbox during registration.
This endpoint is normally called from the signed verification link sent by email. To test it manually, set these `.env` values from that link: `NMRXIV_VERIFY_USER_ID`, `NMRXIV_VERIFY_HASH`, `NMRXIV_VERIFY_EXPIRES`, and `NMRXIV_VERIFY_SIGNATURE`.

In [ ]:
verify_user_id = "user_id" # ID of the user to verify
verify_hash = "verify_hash" # Hash for verification
verify_expires = os.getenv("NMRXIV_VERIFY_EXPIRES") 
verify_signature = os.getenv("NMRXIV_VERIFY_SIGNATURE")

if verify_user_id and verify_hash and verify_expires and verify_signature:
    verify_response = api_request(
        "GET",
        f"/email/verify/{verify_user_id}/{verify_hash}",
        params={"expires": verify_expires, "signature": verify_signature},
    )
    pprint(verify_response)
else:
    print("Set NMRXIV_VERIFY_USER_ID, NMRXIV_VERIFY_HASH, NMRXIV_VERIFY_EXPIRES, and NMRXIV_VERIFY_SIGNATURE to test email verification.")

## Resend email verification link

Sends a new email verification link to the authenticated user's email address. This endpoint can be used when the original verification email was not received or has expired. The user must be authenticated but not yet verified. Requires a Bearer token.

In [ ]:
if "Authorization" in session.headers:
    resend_response = api_request("GET", "/auth/email/resend")
    pprint(resend_response)
else:
    print("Login first to call /auth/email/resend.")

## Register new user account

Creates a new user account in the nmrXiv platform. Supports both regular user registration and ELN (Electronic Lab Notebook) user registration with different verification flows. Regular users receive email verification, while ELN users are auto-verified with a 3-day welcome period.
Set the `NMRXIV_RUN_REGISTRATION=true` in .env to create a real account. Fill the `registration_payload` with unique values before running. The required fields are `first_name`, `last_name`, `email`, `username`, and `password`; `orcid_id` and `affiliation` are optional.

In [ ]:
registration_payload = {
    "first_name": "Dr. Sarah",
    "last_name": "Johnson",
    "email": "sarah.johnson@university.edu",
    "username": "sarah_johnson_chem",
    "password": "SecurePassword123!",
    "orcid_id": "0000-0002-1825-0097" or None,
    "affiliation": "Friedrich Schiller University Jena (FSU, Friedrich-Schiller-Universität Jena) Jena, Germany" or None,
}

run_registration = os.getenv("NMRXIV_RUN_REGISTRATION", "false").lower() == "true"

if run_registration:
    register_response = api_request("POST", "/auth/register", json=registration_payload)
    pprint(register_response)
else:
    print("Set NMRXIV_RUN_REGISTRATION=true in .env to create a real account with registration_payload.")
    pprint({key: value for key, value in registration_payload.items() if key != "password"})

## Revoke current access token and logout user

Invalidates the current Bearer token used for authentication. The user will need to login again to obtain a new token for API access. Run this when you are done with authenticated examples.

In [ ]:
if "Authorization" in session.headers:
    logout_response = api_request("GET", "/auth/logout")
    pprint(logout_response)
    session.headers.pop("Authorization", None)
else:
    print("No Bearer token configured; nothing to logout.")